# Evaluación de los modelos YOLO26 en el conjunto de prueba

Genera `metricas_prueba.csv` (P, R, mAP@0.5 y mAP@0.5:0.95, en total y por clase) y las matrices de confusión de **yolo26n, yolo26s y yolo26m**, con el mismo modo de inferencia que el nodo de borde (de extremo a extremo, sin NMS).

**Cómo usarlo**
1. Menú **Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU (T4)** → Guardar.
2. Menú **Entorno de ejecución → Ejecutar todo**.
3. Cuando lo pida, elija los tres archivos `yolo26n.pt`, `yolo26s.pt` y `yolo26m.pt` de su computadora.
4. Cuando lo pida, pegue su clave de API de Roboflow (Roboflow → Settings → API Keys). No se guarda en ningún archivo.
5. Al final se descarga `evaluacion.zip` con los resultados.

In [ ]:
# 1) Código del sistema y dependencias (versiones del artículo)
!git clone -q https://github.com/updatedtechnology15-collab/Sistema_IoT_granos_cacao.git
%cd Sistema_IoT_granos_cacao
!pip install -q ultralytics==8.4.208 roboflow

In [ ]:
# 2) Subir los tres modelos a la carpeta models/
import os, shutil
from google.colab import files
os.makedirs("models", exist_ok=True)
for nombre in files.upload():
    shutil.move(nombre, os.path.join("models", nombre))
print(sorted(os.listdir("models")))

In [ ]:
# 3) Descargar el dataset de Roboflow (versión 4, la usada en el entrenamiento)
from getpass import getpass
from roboflow import Roboflow
rf = Roboflow(api_key=getpass("Clave de API de Roboflow: "))
ds = rf.workspace("tfm-unir-ensayo-de-corte-cacao").project("cacao-cnn-v1-1-final").version(4).download("yolov8")
print("Dataset en", ds.location)

In [ ]:
# 4) Evaluar los tres modelos en la partición de prueba (una sola vez cada uno)
!python herramientas/evaluar_modelos.py --data {ds.location}/data.yaml --device 0

In [ ]:
# 5) Ver y descargar los resultados
import pandas as pd
display(pd.read_csv("resultados/evaluacion/metricas_prueba.csv"))
shutil.make_archive("evaluacion", "zip", "resultados/evaluacion")
files.download("evaluacion.zip")